# A1 — Corpus Exploration (fill this)
Explore your scanned corpus: page/word counts, scan quality, script and font variety.

In [ ]:
# Single-cell corpus downloader + OCR + explorer
# Usage: open this notebook and run the single cell; it will download the Archive.org item,
# OCR PDFs (if toolchain available), and print summaries.
import os, sys, json, urllib.request, urllib.parse, subprocess, shutil, re, statistics
from pathlib import Path

ITEM = 'ar-raheequl-makhtoom-bangla'
OUTDIR = Path('data') / 'raw' / ITEM
OUTDIR.mkdir(parents=True, exist_ok=True)


def fetch_metadata(item, outdir):
    url = f'https://archive.org/metadata/{item}'
    dest = outdir / 'metadata.json'
    try:
        print('Fetching metadata...')
        with urllib.request.urlopen(url) as r, open(dest, 'wb') as f:
            f.write(r.read())
        with open(dest, 'r', encoding='utf-8') as f:
            return json.load(f)
    except Exception as e:
        print('Failed to fetch metadata:', e)
        return None


meta = fetch_metadata(ITEM, OUTDIR)
if not meta:
    print('No metadata available; aborting.')
else:
    files = meta.get('files', [])
    print('Metadata file count:', len(files))
    # list candidate files
    want_exts = ('.pdf', '.djvu', '.txt', '.zip', '.jpg', '.jpeg', '.png', '.tif', '.tiff')
    candidates = []
    for f in files:
        name = f.get('name')
        if not name:
            continue
        lname = name.lower()
        if any(lname.endswith(e) for e in want_exts) or f.get('format'):
            candidates.append(name)
    print('Download candidates:', len(candidates))
    # download candidates
    for name in candidates:
        url = f'https://archive.org/download/{urllib.parse.quote(ITEM)}/{urllib.parse.quote(name)}'
        outpath = OUTDIR / name
        outpath.parent.mkdir(parents=True, exist_ok=True)
        if outpath.exists():
            print('Skipping existing', name)
            continue
        try:
            print('Downloading', name)
            urllib.request.urlretrieve(url, outpath)
        except Exception as e:
            print('Failed to download', name, e)

    # Summarize files on disk
    all_files = sorted([p for p in OUTDIR.rglob('*') if p.is_file()])
    print('Files on disk:', len(all_files))
    total_bytes = sum(p.stat().st_size for p in all_files)
    print('Total size (MB):', round(total_bytes / 1024 / 1024, 2))
    extcnt = {}
    for p in all_files:
        ext = p.suffix.lower().lstrip('.') or 'noext'
        extcnt[ext] = extcnt.get(ext, 0) + 1
    top = sorted(extcnt.items(), key=lambda x: x[1], reverse=True)[:12]
    print('Top types:')
    for k, v in top:
        print(' ', k, v)
    print('\nSample:')
    for p in all_files[:20]:
        print(' ', p.relative_to(Path.cwd()))

    # OCR PDFs if needed
    pdfs = [p for p in all_files if p.suffix.lower() == '.pdf']
    if not pdfs:
        print('No PDFs to OCR')
    else:
        print('Found', len(pdfs), 'PDFs')
        ocrmypdf = shutil.which('ocrmypdf')
        pdftoppm = shutil.which('pdftoppm')
        tesseract = shutil.which('tesseract')
        pdftotext = shutil.which('pdftotext')
        TESS_LANG = os.environ.get('TESSERACT_LANG', 'ben')
        for pdf in pdfs:
            txtp = pdf.with_suffix('.txt')
            if txtp.exists():
                print('TXT exists, skipping OCR for', pdf.name)
                continue
            print('OCRing', pdf.name)
            done = False
            if ocrmypdf:
                try:
                    tmp = pdf.with_suffix('.ocr.pdf')
                    subprocess.check_call([ocrmypdf, '--skip-text', str(pdf), str(tmp)])
                    if pdftotext:
                        subprocess.check_call([pdftotext, str(tmp), str(txtp)])
                        done = True
                    else:
                        # try extracting text via pypdf if available
                        try:
                            from pypdf import PdfReader
                            r = PdfReader(str(tmp))
                            text = '\n'.join(p.extract_text() or '' for p in r.pages)
                            txtp.write_text(text, encoding='utf-8')
                            done = True
                        except Exception:
                            pass
                except Exception as e:
                    print('ocrmypdf failed:', e)
            if not done and pdftoppm and tesseract:
                try:
                    workdir = pdf.with_suffix('')
                    workdir = workdir.parent / (workdir.name + '_pages')
                    workdir.mkdir(parents=True, exist_ok=True)
                    pref = str(workdir / 'page')
                    subprocess.check_call([pdftoppm, '-png', str(pdf), pref])
                    imgs = sorted(workdir.glob('*.png'))
                    pieces = []
                    for i, img in enumerate(imgs):
                        outbase = str(workdir / f'page_{i}')
                        subprocess.check_call([tesseract, str(img), outbase, '-l', TESS_LANG])
                        tfile = Path(outbase + '.txt')
                        if tfile.exists():
                            pieces.append(tfile.read_text(encoding='utf-8', errors='ignore'))
                    txtp.write_text('\n\n'.join(pieces), encoding='utf-8')
                    done = True
                except Exception as e:
                    print('pdftoppm+tesseract failed:', e)
            if not done:
                print('No OCR toolchain available or OCR failed for', pdf.name)

    # PDF page counts and stats
    def get_pdf_pages(path):
        try:
            from pypdf import PdfReader
            r = PdfReader(str(path))
            return len(r.pages)
        except Exception:
            pass
        try:
            import PyPDF2
            r = PyPDF2.PdfReader(str(path))
            return len(r.pages)
        except Exception:
            pass
        try:
            out = subprocess.check_output(['pdfinfo', str(path)], stderr=subprocess.DEVNULL).decode('utf-8', 'ignore')
            for line in out.splitlines():
                if line.lower().startswith('pages:'):
                    return int(line.split(':', 1)[1].strip())
        except Exception:
            pass
        return None

    pages = []
    for p in pdfs:
        n = get_pdf_pages(p)
        print(p.name, 'pages->', n)
        if n:
            pages.append(n)
    if pages:
        print('PDF pages stats min/median/max:', min(pages), statistics.median(pages), max(pages))

    # Word counts for OCR text files
    txts = sorted([p for p in OUTDIR.rglob('*.txt') if p.is_file()])
    if not txts:
        print('No OCR text files found.')
    else:
        wcounts = []
        for t in txts:
            try:
                text = t.read_text(encoding='utf-8', errors='ignore')
                words = re.findall(r'\w+', text)
                wcounts.append(len(words))
                print(t.name, 'words=', len(words))
            except Exception as e:
                print('Failed reading', t, e)
        if wcounts:
            print('Text files word stats min/median/max:', min(wcounts), statistics.median(wcounts), max(wcounts))